In [1]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
class SupportState(TypedDict):
    messages: Annotated[list, add_messages]
    category: str
    customer_name: str

print("Customer support state created successfully!")

Customer support state created successfully!


In [3]:
def classify_query(state: SupportState):
    query = state["messages"][-1].content.lower()

    if any(word in query for word in ["refund", "payment", "billing"]):
        category = "billing"
    elif any(word in query for word in ["error", "issue", "problem"]):
        category = "technical"
    else:
        category = "general"

    return {"category": category}

print("Classification node created successfully!")

Classification node created successfully!


In [4]:
def support_response(state: SupportState):
    category = state["category"]
    name = state.get("customer_name", "Customer")

    if category == "billing":
        response = f"Hi {name}, I can help with your billing or refund request."
    elif category == "technical":
        response = f"Hi {name}, please describe your technical issue."
    else:
        response = f"Hi {name}, how can I assist you today?"

    return {"messages": [AIMessage(content=response)]}

print("Support response node created successfully!")


Support response node created successfully!


In [5]:
builder = StateGraph(SupportState)

builder.add_node("classify", classify_query)
builder.add_node("respond", support_response)

builder.add_edge(START, "classify")
builder.add_edge("classify", "respond")
builder.add_edge("respond", END)

memory = MemorySaver()
agent = builder.compile(checkpointer=memory)

print("Stateful customer support agent created successfully!")


Stateful customer support agent created successfully!


In [6]:
config = {"configurable": {"thread_id": "customer-001"}}

result = agent.invoke(
    {
        "messages": [HumanMessage(content="I need a refund")],
        "customer_name": "Rahul"
    },
    config=config
)

print("Category:", result["category"])
print("Agent:", result["messages"][-1].content)

Category: billing
Agent: Hi Rahul, I can help with your billing or refund request.


In [7]:
result = agent.invoke(
    {
        "messages": [HumanMessage(content="I have a payment problem")]
    },
    config=config
)

print("Category:", result["category"])
print("Agent:", result["messages"][-1].content)
print("Total messages:", len(result["messages"]))
print("Remembered customer:", result["customer_name"])

Category: billing
Agent: Hi Rahul, I can help with your billing or refund request.
Total messages: 4
Remembered customer: Rahul


In [8]:
for i, query in enumerate(["My app has an error", "Hello"]):
    result = agent.invoke(
        {
            "messages": [HumanMessage(content=query)],
            "customer_name": "Priya"
        },
        config={"configurable": {"thread_id": f"test-{i}"}}
    )

    print("Query:", query)
    print("Category:", result["category"])
    print("Agent:", result["messages"][-1].content)
    print()

Query: My app has an error
Category: technical
Agent: Hi Priya, please describe your technical issue.

Query: Hello
Category: general
Agent: Hi Priya, how can I assist you today?



session_a = agent.get_state(config).values

session_b = agent.get_state(
    {"configurable": {"thread_id": "test-0"}}
).values

assert session_a["customer_name"] == "Rahul"
assert session_b["customer_name"] == "Priya"
assert len(session_a["messages"]) == 4
assert len(session_b["messages"]) == 2

print("Session isolation test passed!")## W10D5 - Stateful Customer Support Agent

### Test Results
- Billing support: Passed
- Technical support: Passed
- General support: Passed
- Conversation history retention: Passed
- Customer name retention: Passed

### Conclusion
Successfully implemented a stateful customer support
agent using LangGraph and MemorySaver.

The agent classifies customer queries and maintains
conversation history using unique thread IDs.

### Limitations
- Uses rule-based classification and predefined responses.
- Memory is stored in-process, not permanently.

session_a = agent.get_state(config).values

session_b = agent.get_state(
    {"configurable": {"thread_id": "test-0"}}
).values

assert session_a["customer_name"] == "Rahul"
assert session_b["customer_name"] == "Priya"
assert len(session_a["messages"]) == 4
assert len(session_b["messages"]) == 2

print("Session isolation test passed!")

In [9]:
session_a = agent.get_state(config).values

session_b = agent.get_state(
    {"configurable": {"thread_id": "test-0"}}
).values

assert session_a["customer_name"] == "Rahul"
assert session_b["customer_name"] == "Priya"
assert len(session_a["messages"]) == 4
assert len(session_b["messages"]) == 2

print("Session isolation test passed!")

Session isolation test passed!
